# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring a scientific dataset using the `mlcroissant` library, referencing all elements such as record sets, fields, and columns by their `@id` fields for precision and reproducibility.

### Dataset Source
The dataset source is a Croissant schema available at a public URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We'll load the metadata and available record sets from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata information
meta = dataset.metadata
print(f"Dataset Name: {getattr(meta, 'name', None)}\n")
print(f"Description: {getattr(meta, 'description', None)}\n")
print(f"Version: {getattr(meta, 'version', None)}\n")
print(f"Published: {getattr(meta, 'datePublished', None)}\n")

## 2. Data Overview
Let's enumerate the available record sets and their referenced field `@id`s. This information will inform our selection for extraction and downstream analysis. All record set and field identifiers are referenced by their exact `@id` from the schema.

In [ ]:
# List all record sets and their fields using their @id
record_sets = [rs for rs in getattr(meta, 'recordSet', [])]
if not record_sets:
    print("No record sets found in the top-level metadata. Attempting to infer record sets from data files..\n")
    # Alternative: Try to enumerate them via metadata.distribution or check automatic population by mlcroissant
    distributions = getattr(meta, 'distribution', [])
    print("Distributions found in this dataset:\n")
    for dist in distributions:
        print(f" - Distribution @id: {getattr(dist, '@id', dist)}")
    print("\nSee https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json for full details.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {getattr(rs, '@id', rs)}")
        fields = getattr(rs, 'field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"  Field @id: {getattr(field, '@id', field)}")
        print("")

## 3. Data Extraction
We'll now extract data from a specific record set, referencing both the record set and fields by their `@id`. Note that Croissant-compliant datasets sometimes have one main record set.

**NOTE**: If no explicit `recordSet` is provided at the schema top-level, mlcroissant typically registers them by their `@id` as listed in the schema's RecordSet objects. In this dataset, you should inspect the record set `@id`s and their fields using the outputs above before loading the data.

For this dataset, we will use the main data record set whose `@id` is `'cr:RecordSet_1'` for this demonstration (replace if schema provides different or additional `@id`s).

In [ ]:
# Define the record set `@id` based on the schema (example shown - replace with actual @id from the schema overview)
RECORD_SET_ID = 'cr:RecordSet_1'  # Replace with actual @id after inspecting the schema

try:
    records = list(dataset.records(record_set=RECORD_SET_ID))
except Exception as e:
    print(f"Could not load records for {RECORD_SET_ID}. Possible reasons: record set does not exist or the data is referenced differently.\nException: {e}")
    records = []

if records:
    df = pd.DataFrame(records)
    print(f"Columns (@id): {df.columns.tolist()}")
    display(df.head())
else:
    print(f"No records loaded for record set {RECORD_SET_ID}.")

## 4. Exploratory Data Analysis (EDA)
Let's process the extracted data. We'll demonstrate typical analysis steps such as filtering on a numeric field (e.g. `cr:coverage`), normalizing, and grouping by another field (e.g. `cr:description`). All columns are referenced by their schema `@id`. You may adapt field IDs based on the schema content.

In [ ]:
# Replace these example field @id's with actual ones from your inspection
NUMERIC_FIELD_ID = 'cr:coverage'  # e.g. coverage percentage
GROUP_FIELD_ID = 'cr:description'  # e.g. protein description

if records and NUMERIC_FIELD_ID in df.columns:
    # Convert the numeric field to numeric type for operations (if necessary, e.g. float)
    df[NUMERIC_FIELD_ID] = pd.to_numeric(df[NUMERIC_FIELD_ID], errors='coerce')

    # Remove NaN, filter for coverage > 10
    threshold = 10
    filtered_df = df[df[NUMERIC_FIELD_ID] > threshold].copy()

    print(f"Filtered records where {NUMERIC_FIELD_ID} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{NUMERIC_FIELD_ID}_normalized"] = (
        filtered_df[NUMERIC_FIELD_ID] - filtered_df[NUMERIC_FIELD_ID].mean()
    ) / filtered_df[NUMERIC_FIELD_ID].std()

    print(f"Normalized {NUMERIC_FIELD_ID} for filtered records:")
    display(filtered_df[[NUMERIC_FIELD_ID, f"{NUMERIC_FIELD_ID}_normalized"]].head())

    # Group by key attribute and calculate summary
    if GROUP_FIELD_ID in filtered_df.columns:
        grouped_df = filtered_df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean().to_frame("mean_coverage")
        print(f"Grouped mean {NUMERIC_FIELD_ID} by {GROUP_FIELD_ID}:")
        display(grouped_df.head())
else:
    print(f"Field {NUMERIC_FIELD_ID} not found in records for EDA or records unavailable.")

## 5. Visualization
Let's visualize the distribution of the coverage field and the normalized field. All fields are referenced by their schema `@id`.

_Requires matplotlib. If not already installed, run `!pip install matplotlib`._

In [ ]:
import matplotlib.pyplot as plt

if records and NUMERIC_FIELD_ID in df.columns:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    df[NUMERIC_FIELD_ID].dropna().hist(bins=30)
    plt.title(f"Distribution of {NUMERIC_FIELD_ID}")
    plt.xlabel(NUMERIC_FIELD_ID)
    plt.ylabel("Count")

    if f"{NUMERIC_FIELD_ID}_normalized" in filtered_df.columns:
        plt.subplot(1,2,2)
        filtered_df[f"{NUMERIC_FIELD_ID}_normalized"].dropna().hist(bins=30, color='orange')
        plt.title(f"Normalized {NUMERIC_FIELD_ID} (filtered)")
        plt.xlabel(f"{NUMERIC_FIELD_ID}_normalized")
        plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization. Run previous cells and ensure data loaded correctly.")

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load Croissant-compliant metadata and records using `mlcroissant`
- Reference all data entities by their `@id` for reproducibility
- Explore, filter, normalize, and visualize a typical record set from the FAIR² dataset
- Prepare the ground for more advanced domain-specific analysis

**Note:** For deeper analysis, review the Croissant schema for more field details, complex entities, and join relationships. Make sure to always reference the `@id` field for precise handling of schema objects in FAIR data science workflows.